# Parameter Golf - Google Colab

OpenAI Model Craft Challenge: Parameter Golf

**ランタイム設定**: 「ランタイム」→「ランタイムのタイプを変更」→ **T4 GPU** を選択

## 1. GPU確認

In [ ]:
!nvidia-smi

## 2. セットアップ（クローン・依存関係・データDL・T4パッチ）

In [ ]:
import os, shutil

# クリーンクローン
if os.path.exists('/content/parameter-golf'):
    shutil.rmtree('/content/parameter-golf')
!git clone https://github.com/tsubasagit/parameter-golf.git /content/parameter-golf
%cd /content/parameter-golf
!pip install -q sentencepiece huggingface-hub datasets tqdm zstandard

In [ ]:
# データセットDL（最小構成: 1 shard）
!python3 data/cached_challenge_fineweb.py --variant sp1024 --train-shards 1

In [ ]:
# ===== T4 GPU互換パッチ =====
# T4はbfloat16 Flash Attentionとenable_gqaをサポートしていないため、
# 全train_gpt.pyにパッチを適用する

import glob, re

def patch_for_t4(filepath):
    with open(filepath, 'r') as f:
        code = f.read()
    original = code

    # 1. enable_gqa を手動K/Vヘッド展開に置換
    # パターン: SDPAにenable_gqaがある場合、手前にrepeat_interleaveを挿入
    code = re.sub(
        r'(        q = q \* self\.q_gain\.to\(dtype=q\.dtype\)\[None, :, None, None\]\n)'
        r'(        y = F\.scaled_dot_product_attention\(\n)'
        r'(            q, k, v,.*?\n)'
        r'(.*?enable_gqa.*?\n)'
        r'(        \))',
        r'\1'
        r'        if self.num_kv_heads != self.num_heads:\n'
        r'            rep = self.num_heads // self.num_kv_heads\n'
        r'            k = k.repeat_interleave(rep, dim=1)\n'
        r'            v = v.repeat_interleave(rep, dim=1)\n'
        r'\2\3        )',
        code,
        flags=re.DOTALL
    )
    # もっとシンプルなパターンも試す（スタイル違い対応）
    if 'enable_gqa' in code:
        code = re.sub(
            r'(        y = F\.scaled_dot_product_attention\(\n'
            r'            q,\n'
            r'            k,\n'
            r'            v,\n'
            r'            attn_mask=None,\n'
            r'            is_causal=True,\n'
            r'            enable_gqa=.*?,\n'
            r'        \))',
            '        if self.num_kv_heads != self.num_heads:\n'
            '            rep = self.num_heads // self.num_kv_heads\n'
            '            k = k.repeat_interleave(rep, dim=1)\n'
            '            v = v.repeat_interleave(rep, dim=1)\n'
            '        y = F.scaled_dot_product_attention(\n'
            '            q,\n'
            '            k,\n'
            '            v,\n'
            '            attn_mask=None,\n'
            '            is_causal=True,\n'
            '        )',
            code
        )
    # ワンライナーパターン
    if 'enable_gqa' in code:
        code = re.sub(
            r'        y = F\.scaled_dot_product_attention\(\n'
            r'            q, k, v, attn_mask=None, is_causal=True,\n'
            r'            enable_gqa=.*?,\n'
            r'        \)',
            '        if self.num_kv_heads != self.num_heads:\n'
            '            rep = self.num_heads // self.num_kv_heads\n'
            '            k = k.repeat_interleave(rep, dim=1)\n'
            '            v = v.repeat_interleave(rep, dim=1)\n'
            '        y = F.scaled_dot_product_attention(\n'
            '            q, k, v, attn_mask=None, is_causal=True,\n'
            '        )',
            code
        )

    # 2. SDP backend: flash only → flash + mem_efficient + math
    code = code.replace(
        'enable_flash_sdp(True)\n    enable_mem_efficient_sdp(False)\n    enable_math_sdp(False)',
        'enable_flash_sdp(False)\n    enable_mem_efficient_sdp(True)\n    enable_math_sdp(True)'
    )
    # 別のインデントパターン
    code = code.replace(
        'enable_flash_sdp(True)\n    enable_mem_efficient_sdp(False)\n    enable_math_sdp(False)',
        'enable_flash_sdp(False)\n    enable_mem_efficient_sdp(True)\n    enable_math_sdp(True)'
    )

    # 3. bfloat16 → float16
    code = code.replace('.bfloat16()', '.half()')
    code = code.replace('dtype=torch.bfloat16', 'dtype=torch.float16')

    if code != original:
        with open(filepath, 'w') as f:
            f.write(code)
        print(f'  Patched: {filepath}')
    else:
        print(f'  No changes needed: {filepath}')

# ベースラインと全recordsのtrain_gpt.pyにパッチ適用
targets = ['train_gpt.py'] + glob.glob('records/**/train_gpt*.py', recursive=True)
print(f'Patching {len(targets)} files for T4 compatibility...')
for f in targets:
    patch_for_t4(f)

# 検証: enable_gqaが残っていないことを確認
import subprocess
result = subprocess.run(['grep', '-rn', 'enable_gqa', 'train_gpt.py'], capture_output=True, text=True)
if result.stdout:
    print(f'WARNING: enable_gqa still found in train_gpt.py:\n{result.stdout}')
else:
    print('OK: enable_gqa removed from train_gpt.py')

result2 = subprocess.run(['grep', '-n', 'flash_sdp(True)', 'train_gpt.py'], capture_output=True, text=True)
if result2.stdout:
    print(f'WARNING: flash_sdp still enabled:\n{result2.stdout}')
else:
    print('OK: flash_sdp disabled for T4')

---
## 3. ベースライン実行

In [ ]:
import os
os.environ['RUN_ID'] = 'colab_baseline'
os.environ['ITERATIONS'] = '500'
os.environ['TRAIN_BATCH_TOKENS'] = '131072'
os.environ['VAL_LOSS_EVERY'] = '100'
os.environ['VAL_BATCH_SIZE'] = '65536'
os.environ['MAX_WALLCLOCK_SECONDS'] = '600'

!torchrun --standalone --nproc_per_node=1 train_gpt.py

---
## 4. 改良版: 上位テクニック適用

リーダーボード2位（1.1458 BPB）のスクリプトを使用。

| テクニック | 効果 |
|-----------|------|
| MLP 3x拡張 | hidden dim 1024→1536 |
| SmearGate | 前トークンとのゲート融合 |
| BigramHash(4096) | トークンペアのハッシュ埋め込み |
| U-Net Skip | エンコーダ→デコーダのスキップ接続 |
| SWA | 学習後半のチェックポイント平均 |
| Muon WD | Weight Decay追加 |
| Sliding Window Eval | stride=64で評価精度向上 |

In [ ]:
import shutil
src = '/content/parameter-golf/records/track_10min_16mb/2026-03-20_Int6_MLP3x_SmearGate_BigramHash_MuonWD_SWA/train_gpt.py'
dst = '/content/parameter-golf/train_gpt_improved.py'
shutil.copy2(src, dst)
print(f'Copied improved script (already patched for T4)')

In [ ]:
import os

os.environ['RUN_ID'] = 'colab_improved_v1'
os.environ['DATA_PATH'] = './data/datasets/fineweb10B_sp1024'
os.environ['TOKENIZER_PATH'] = './data/tokenizers/fineweb_1024_bpe.model'
os.environ['VOCAB_SIZE'] = '1024'

os.environ['ITERATIONS'] = '500'
os.environ['MAX_WALLCLOCK_SECONDS'] = '900'
os.environ['TRAIN_BATCH_TOKENS'] = '65536'
os.environ['VAL_BATCH_SIZE'] = '65536'
os.environ['TRAIN_SEQ_LEN'] = '1024'
os.environ['VAL_LOSS_EVERY'] = '100'

os.environ['NUM_LAYERS'] = '9'
os.environ['MODEL_DIM'] = '512'
os.environ['NUM_HEADS'] = '8'
os.environ['NUM_KV_HEADS'] = '4'
os.environ['MLP_MULT'] = '3'
os.environ['MATRIX_LR'] = '0.02'
os.environ['SCALAR_LR'] = '0.02'
os.environ['MUON_MOMENTUM'] = '0.99'
os.environ['WARMDOWN_ITERS'] = '200'

!torchrun --standalone --nproc_per_node=1 train_gpt_improved.py

---
## 5. 結果比較

| 実行 | 期待 val_bpb | 備考 |
|------|-------------|------|
| ベースライン | ~1.3+ | 9L, MLP2x, 500iter, T4 |
| 改良版 | ~1.25前後 | SmearGate+BigramHash+MLP3x |
| SOTA (8xH100) | 1.1428 | 全テクニック, 20000iter |

---
## 6. カスタム実験

In [ ]:
import os

os.environ['RUN_ID'] = 'colab_experiment_custom'
os.environ['DATA_PATH'] = './data/datasets/fineweb10B_sp1024'
os.environ['TOKENIZER_PATH'] = './data/tokenizers/fineweb_1024_bpe.model'
os.environ['VOCAB_SIZE'] = '1024'
os.environ['ITERATIONS'] = '500'
os.environ['MAX_WALLCLOCK_SECONDS'] = '900'
os.environ['TRAIN_BATCH_TOKENS'] = '65536'
os.environ['VAL_BATCH_SIZE'] = '65536'
os.environ['VAL_LOSS_EVERY'] = '100'

# --- ここを変更 ---
os.environ['NUM_LAYERS'] = '10'
os.environ['TRAIN_SEQ_LEN'] = '1024'
os.environ['MLP_MULT'] = '3'
os.environ['MATRIX_LR'] = '0.02'
os.environ['MUON_MOMENTUM'] = '0.99'
os.environ['WARMDOWN_ITERS'] = '200'

!torchrun --standalone --nproc_per_node=1 train_gpt_improved.py